# Tensor-train decomposition and completion lab
Inspect TT cores, compare rank caps, and run a data-consistent completion experiment. Use held-out observations for predictive evaluation.

In [ ]:
import sys
sys.path.insert(0, "implementations/python/tensor-decomposition")
import numpy as np
from tt_svd import tt_svd, tt_reconstruct, tt_complete
rng = np.random.default_rng(42)


In [ ]:
a, b, c, d = (rng.normal(size=n) for n in (8, 7, 6, 5))
X = np.einsum('i,j,k,l->ijkl', a, b, c, d)
exact = tt_svd(X)
approx = tt_svd(X, max_rank=1)
exact['ranks'], exact['relative_error'], approx['relative_error']


In [ ]:
cores = exact['cores']
np.linalg.norm(tt_reconstruct(cores) - X) / np.linalg.norm(X)


In [ ]:
mask = rng.random(X.shape) > 0.3
observed = X.copy()
observed[~mask] = np.nan
result = tt_complete(observed, mask, max_rank=1, max_iter=100, tolerance=1e-7)
np.max(np.abs(result['completed'][mask] - X[mask])), result['iterations']


## Evaluate honestly
The completion routine restores observed values exactly, so observed RMSE is zero by construction. To estimate prediction quality, hide a second random subset of known values, fit using the remaining observations, and evaluate RMSE only on the held-out subset. Rank and initialization must be selected without looking at the held-out values.